<a href="https://colab.research.google.com/github/claramanolache/ML_Intro/blob/main/Week_5_Practice.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Week 5: Practice Notebook

**You do not need to submit this Notebook.** It is for practice only.

## Introduction to the Dataset

The Breast Cancer Wisconsin (Diagnostic) dataset is one of the most widely used datasets in machine learning for demonstrating classification techniques. It was created from digitized images of fine needle aspirate samples of breast tissue, where features describing the characteristics of cell nuclei were extracted. These 30 numerical features capture details such as the size, texture, shape, and smoothness of the cells, helping to distinguish between malignant (cancerous) and benign (non-cancerous) tumors. The dataset contains 569 total samples, with 212 labeled malignant and 357 labeled benign. Because the data is relatively clean and well-balanced, it serves as an excellent resource for learning, testing, and comparing machine learning models.

### Goal
Explore and apply support vector machines (SVMs) for both classification and regression using the Breast Cancer Wisconsin dataset.

## Import Libraries

In [ ]:
import itertools  # Used for creating pair-wise combinations.

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn import datasets
from sklearn import dummy
from sklearn import metrics
from sklearn import model_selection
from sklearn import preprocessing
from sklearn import svm

## Load the Dataset

In [ ]:
data = datasets.load_breast_cancer(as_frame=True)
X, y = data.data, data.target

### Inspect the Data Types

As can be seen below, features in this dataset are real numbers.

In [ ]:
X.info()

In [ ]:
X.head()

The labels in the data are either 0 (malignant) or 1 (benign).

In [ ]:
pd.Series(data.target_names).rename("label")

## Preparing the Data

The following variable will be used to set a random seed for certain processes. Using a seed makes random processes deterministic, meaning this notebook will do things like split data the same way on each run. You are free to change this value to explore how the notebook runs differently.

In [ ]:
seed = 42

### Split into Training and Test Sets

In [ ]:
X_train, X_test, y_train, y_test = model_selection.train_test_split(
    X, y, test_size=0.2, random_state=seed
)

### Scale the Features

Support vector machines require scaled features to find good margins.

Because StandardScaler returns a NumPy array, here we pass the scaled features back into a DataFrame constructor to retain the conveniences of Pandas.

In [ ]:
scaler = preprocessing.StandardScaler()
X_train = pd.DataFrame(scaler.fit_transform(X_train), columns=X.columns)
X_test = pd.DataFrame(scaler.transform(X_test), columns=X.columns)

## Find the Least Linearly Separable Feature Pairs

The code below iterates through every pair of features (using the itertools module) and attempts to fit a linear SVM with only two features at a time to make class predictions. The feature pairs that result in the worst model scores are considered the least linearly separable here; the worst 20 such pairs are displayed afterward.

In [ ]:
scores = []
for f1, f2 in itertools.combinations(X_train.columns, 2):
    X_pair = X_train[[f1, f2]]
    clf = svm.LinearSVC(C=1.0, max_iter=10000, random_state=seed)
    avg_score = model_selection.cross_val_score(
        clf,
        X_pair,
        y_train,
        cv=3
    ).mean()
    scores.append({"features": (f1, f2), "accuracy": avg_score})
df_scores = pd.DataFrame(scores).sort_values("accuracy").reset_index(drop=True)
df_scores.head(20)

### Plot the Feature Pairs

Now we will plot the pairs of features on which the linear SVM scored lowest. The two classes - benign and malignant - are color-coded.

In [ ]:
top_pairs = df_scores.head(20)["features"]
_, axes = plt.subplots(4, 5, figsize=(20, 16))
axes = axes.flatten()
for ax, (f1, f2) in zip(axes, top_pairs):
    ax.scatter(X_train[f1], X_train[f2], c=y_train, cmap="viridis", alpha=0.7)
    ax.set_title(f"{f1} vs {f2}")
    ax.set_xticks([])
    ax.set_yticks([])
plt.tight_layout()
plt.show()

From these plots, it is easy to see why the linear SVM performed poorly.

Now let's see if a SVM using the RBF (Radial Basis Function) kernel can do any better.

#### Self-Check

In the above plots, which pairs of features appear to be the least linearly correlated? Note that this is different from asking which plot demonstrates the least linear separability between the two classes.

## Exploring the Regularization Parameter C

The C parameter controls the trade-off between maximizing the margin and minimizing classification errors.

### Select a Feature Pair

Choose two features to explore based on the plots above. We will try to build a non-linear SVM model that performs better than the previous linear SVM model.

In [ ]:
# Replace these features with any pair of your choice.
feature_pair = ["smoothness error", "fractal dimension error"]

We are going to use `GridSearchCV` to find the best combination of hyperparameters on the training data. This object collects all the hyperparameter options we provide it and tries every combination of them on the given model (SVC in this case), using cross-validation to determine which combination of hyperparameters performed the best.

The variables defined below will be used as the hyperparameter options for `GridSearchCV`. Try changing their range and see if you find better hyperparameter combinations for the selected feature pair. Note that increasing the number of these options will cause the search to take longer.

In [ ]:
params_c = [0.001, 0.01, 0.1, 1.0, 10.0, 100.0, 1000.0]
params_gamma = [0.001, 0.01, 0.1, 1.0, 10.0, 100.0, 1000.0]

In [ ]:
X_pair = X_train[feature_pair]
clf = model_selection.GridSearchCV(
    # The model that we will use to explore different hyperparameters.
    estimator=svm.SVC(kernel="rbf"),
    # A dictionary that defines which values to try for each hyperparameter.
    # The hyperparameter name (as a string) is the dictionary key,
    # The key's value is a list of values to try in sequence.
    param_grid={
        "C": params_c,
        "gamma": params_gamma
    },
    # The number of cross-validation folds to use when evaluating the model
    # on a particular set of hyperparameter values.
    cv=5
)
clf.fit(X_pair, y_train)
clf.best_params_

The table below shows how each combination of C and gamma scored during cross-validation during the grid search.

In [ ]:
results = pd.DataFrame(
    clf.cv_results_,
    columns=["param_C", "param_gamma", "mean_test_score"]
).sort_values("mean_test_score", ascending=False).reset_index(drop=True)
results.head()

The heatmap below plots C and gamma on the axes, with color used to indicate the accuracy for each pair of hyperparameter values.

In [ ]:
results = clf.cv_results_
scores_mean = results["mean_test_score"].reshape(len(params_c), len(params_gamma))

plt.figure(figsize=(10, 8))
plt.imshow(scores_mean, interpolation="nearest", cmap=plt.cm.viridis)
plt.xlabel("gamma")
plt.ylabel("C")
plt.colorbar(label="Mean Test Accuracy")
plt.xticks(np.arange(len(params_gamma)), params_gamma)
plt.yticks(np.arange(len(params_c)), params_c)
plt.title("Mean Test Accuracy for Model Hyperparameters")
plt.show()

Before moving on, you are encouraged to try different pairs of features (refer to the earlier plot of non-linearly separable feature pairs) and explore the impact of different regularization hyperparameters on the non-linear SVM model.

## Support Vector Classification

Now let's turn our attention to classifying samples as benign or malignant.

### Baseline

Before we try a SVM classifier, let's set a baseline with `DummyClassifier`.

In [ ]:
model_selection.cross_val_score(
    dummy.DummyClassifier(),
    X_train,
    y_train,
    cv=3
).mean()

Recall that, by default, the dummy classifier just predicts the most common label.

In [ ]:
dummy_clf = dummy.DummyClassifier()
dummy_clf.fit(X_train, y_train)
dummy_clf.predict(X_test)

### Linear SVM

Let's train a linear SVM so that we can compare it with our non-linear SVM.

In [ ]:
score = model_selection.cross_val_score(
    svm.LinearSVC(C=1.0),
    X_train,
    y_train,
    cv=3
).mean()
print(f"Average accuracy in cross-validation: {score:.3f}")

In [ ]:
lin_clf = svm.LinearSVC(C=1.0)
lin_clf.fit(X_train, y_train)
lin_pred = lin_clf.predict(X_test)

### RBF-Kernel SVM

Finally, let's train the non-linear SVM model. Note that a `gamma` value of "scale" means the amount of regularization is based on the number of features. See the [documentation](https://scikit-learn.org/stable/modules/generated/sklearn.svm.SVC.html) for details.

In [ ]:
score = model_selection.cross_val_score(
    svm.SVC(kernel="rbf", C=1.0, gamma="scale"),
    X_train,
    y_train,
    cv=3
).mean()
print(f"Average accuracy in cross-validation: {score:.3f}")

In [ ]:
svc_clf = svm.SVC(kernel="rbf", C=1.0, gamma="scale")
svc_clf.fit(X_train, y_train)
svc_pred = svc_clf.predict(X_test)

### Confusion Matrices

The two confusion matrices below provide a comparison of the linear and non-linear SVM models.

Since both models scored quite high, the matrices are very similar; the only difference is that the linear SVM predicted a few benign samples as malignant.

These results may change if you change the regularization hyperparameters above - which you are encouraged to do!

In [ ]:
cm_lin = metrics.confusion_matrix(y_test, lin_pred)
cm_svc = metrics.confusion_matrix(y_test, svc_pred)

_, axes = plt.subplots(1, 2, figsize=(12, 5))

disp_lin = metrics.ConfusionMatrixDisplay(confusion_matrix=cm_lin)
disp_lin.plot(ax=axes[0], cmap=plt.cm.Blues)
axes[0].set_title("LinearSVC Confusion Matrix")

disp_svc = metrics.ConfusionMatrixDisplay(confusion_matrix=cm_svc)
disp_svc.plot(ax=axes[1], cmap=plt.cm.Blues)
axes[1].set_title("SVC (RBF) Confusion Matrix")

plt.tight_layout()
plt.show()

#### Self-Check

For this task, are we concerned with higher precision or recall with respect to classifying malignant samples? Did the non-linear SVM perform any better than the linear SVM in this regard? See if you are able to improve on these intial results.

### Support Vector Regression

Although the Breast Cancer dataset is for classification, you can experiment with regression by predicting one feature with respect to another feature.

As before, choose a pair of features from the plot at the start. This time, one feature will be used to estimate the other.

In [ ]:
feature_pair = ["smoothness error", "fractal dimension error"]

In [ ]:
X_train_reg = X_train[feature_pair[0:1]]
y_train_reg = X_train[feature_pair[1]]
X_test_reg = X_test[feature_pair[0:1]]
y_test_reg = X_test[feature_pair[1]]

In [ ]:
lin_svr = svm.LinearSVR(C=0.1, max_iter=10000, random_state=seed)
lin_svr.fit(X_train_reg, y_train_reg)
y_pred_linear = lin_svr.predict(X_test_reg)
print("LinearSVR MSE:", metrics.mean_squared_error(y_test_reg, y_pred_linear))

In [ ]:
svr_rbf = svm.SVR(kernel="rbf", C=0.1, gamma="scale")
svr_rbf.fit(X_train_reg, y_train_reg)
y_pred_rbf = svr_rbf.predict(X_test_reg)
print("SVR (RBF) MSE:", metrics.mean_squared_error(y_test_reg, y_pred_rbf))

## What's Next?

This Notebook provides an introduction to using SVMs with scikit-learn. See pages in Canvas from this week's module for more information on how SVMs work and why tuning hyperparameters can make a difference.

